# nb_07_silver_claims

**Layer**: Silver | **Source**: `lh_bronze_insurance.claims_raw` | **Target**: `lh_silver_insurance.claims`

**Data Quality Rules**:
- Cast `date_of_loss`, `date_filed` to DateType; `estimated_amount` to DoubleType
- Require non-null: `claim_id`, `policy_id`, `claim_status`
- Deduplicate on `claim_id` (keep latest `_ingested_at`)
- Standardize `claim_type`, `claim_status` to lowercase
- Rejects → `claims_quarantine`

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit, row_number, when, coalesce, lower
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_07_silver_claims").getOrCreate()

print("nb_07_silver_claims - Silver Layer Transformation")

In [ ]:
# ============================================================
# Step 1: Read from Bronze
# ============================================================
df_bronze = spark.table("claims_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count:,}")
df_bronze.printSchema()

In [ ]:
# ============================================================
# Step 2: Cast types & clean
# ============================================================
df_typed = df_bronze \
    .withColumn("claim_id", trim(col("claim_id"))) \
    .withColumn("policy_id", trim(col("policy_id"))) \
    .withColumn("date_of_loss", to_date(col("date_of_loss"), "yyyy-MM-dd")) \
    .withColumn("date_filed", to_date(col("date_filed"), "yyyy-MM-dd")) \
    .withColumn("claim_type", lower(trim(col("claim_type")))) \
    .withColumn("claim_status", lower(trim(col("claim_status")))) \
    .withColumn("estimated_amount", col("estimated_amount").cast(DoubleType())) \
    .withColumn("description", trim(col("description")))

print("Type casting complete.")

In [ ]:
# ============================================================
# Step 3: Validate & route rejects
# ============================================================
REQUIRED_FIELDS = ["claim_id", "policy_id", "claim_status"]

rejection_conditions = [
    when((col(f).isNull()) | (col(f) == ""), lit(f"null_{f}"))
    for f in REQUIRED_FIELDS
]

df_validated = df_typed.withColumn("_rejection_reason", coalesce(*rejection_conditions))

df_valid = df_validated.filter(col("_rejection_reason").isNull()).drop("_rejection_reason")
df_rejected = df_validated.filter(col("_rejection_reason").isNotNull())

valid_count = df_valid.count()
rejected_count = df_rejected.count()
print(f"Valid: {valid_count:,} | Rejected: {rejected_count:,}")

In [ ]:
# ============================================================
# Step 4: Deduplicate on claim_id
# ============================================================
window_spec = Window.partitionBy("claim_id").orderBy(col("_ingested_at").desc())

df_deduped = df_valid \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

deduped_count = df_deduped.count()
dupes_removed = valid_count - deduped_count
print(f"After dedup: {deduped_count:,} ({dupes_removed} duplicates removed)")

In [ ]:
# ============================================================
# Step 5: Write Silver & Quarantine
# ============================================================
silver_columns = ["claim_id", "policy_id", "date_of_loss", "date_filed",
                  "claim_type", "claim_status", "estimated_amount", "description"]

df_silver = df_deduped.select(silver_columns) \
    .withColumn("_processed_at", current_timestamp())

df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("claims")
print(f"✓ {deduped_count:,} rows → claims")

if rejected_count > 0:
    df_rejected.withColumn("_quarantined_at", current_timestamp()) \
        .write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("claims_quarantine")
    print(f"✓ {rejected_count:,} rows → claims_quarantine")

In [ ]:
# ============================================================
# Validation
# ============================================================
print("\nSILVER CLAIMS - SUMMARY")
print("=" * 50)
print(f"  Bronze input:       {bronze_count:>8,}")
print(f"  Rejected:           {rejected_count:>8,}")
print(f"  Duplicates removed: {dupes_removed:>8,}")
print(f"  Silver output:      {deduped_count:>8,}")
print(f"  Quality rate:       {(deduped_count/bronze_count*100):.1f}%")

assert spark.table("claims").count() == deduped_count
print("\n✓ Verification passed")
spark.table("claims").show(5, truncate=25)